# 노드 B — eval (tonight)

**2026-08-24 밤 · 6 GPU = 3노드 x 2 GPU. 이 노트북은 노드 B 전용.**

세 노드 전부 `exp5_tonight.py` 하나만 부른다. 잡 정의 · 우선순위 · 스킵 · 집계가 전부 거기 있다.

| 노드 | GPU | 역할 | 오늘 밤 산출 |
|---|---|---|---|
| **A** | 0,1 | 학습 2잡 (~15h) | 0셀 → **내일 오후 해금** |
| **B** | 0,1 | eval (우선순위 짝수) | ~12셀 |
| **C** | 0,1 | eval (우선순위 홀수) | ~12셀 |

역할은 고정이 아니다 — 3)번 `suggest()`가 인벤토리를 보고 조정한다
(ready eval 큐가 얕으면 그 GPU를 학습으로 돌린다).

**예산**: eval 1셀(LIBERO-10 x 50ep/task = 500ep) ≈ 2 GPU-h · 학습 1잡(150k) ≈ 15 GPU-h
(해준님 8/11 실측 "500ep 2시간, 5000ep 하루" 기준)

---

## 이 노드가 하는 일

기본은 **ckpt가 이미 있는 eval 셀만** 우선순위대로 돈다 (`우선순위 짝수 index`). eval 12셀이 통째로 밀리므로
**eval이 남아 있는 동안은 학습을 걸지 말 것.**
단 3)번 `suggest()`가 이 노드를 `train`으로 배정하면 8)번 학습 셀을 쓴다
(ready eval 큐가 얕은데 학습이 밀려 있는 경우 — 학습만이 내일 이후의 eval을 열어준다).

**우선순위**

1. **TE 축** (stride=1 + coeff 0.01) — 크로스오버 K 찍기. K=100은 이미 측정돼 있다
   (ACT+TE 30.5 vs BiMamba+TE 49.3 = **+18.8**). K=50에서 ACT+TE가 이기고 K=100에서 뒤집히면
   그게 논문 그림 1이다. **TE는 재학습이 필요 없다** — ACT ckpt에 플래그만 얹는다.
2. **K=100 레짐 맵**, 긴 stride부터 (8/18 가설: BiMamba는 긴 stride에서 산다)
3. **K=50 레짐 맵**, (8/18 가설: carry는 짧은 stride에서 산다)

`_has_valid()` 스킵이 있으니 **아침에 그냥 재실행하면 남은 것부터 이어서** 돈다.



## 1) 부팅

In [ ]:
import sys
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()          # common_final reload + 태그 등록 (순서 중요)

# 이 노드에서 쓸 GPU. 한 노드에서 창을 2개 띄울 때만 [2, 3] 처럼 직접 지정.
GPUS = v23.available_gpus([0, 1])
print('GPUS =', GPUS)


## 2) 인벤토리

세 노드가 같은 FS를 보므로 A와 같은 표가 나온다. `MISS`인 태그를 참조하는 eval 셀은 자동으로 큐에서 빠진다(노드 A가 학습한 뒤 잡힌다).

In [ ]:
rows = X.inventory()


## 3) 역할 배정

eval은 ckpt가 있어야 돌아간다. **ready eval 큐가 얕으면 그 GPU는 학습에 주는 게 맞다.** `suggest()`가 인벤토리를 보고 정해준다 — 출력이 "기본값과 다르다"면 알려주는 `X.ROLE_OVERRIDE = ...` 한 줄을 **세 노트북 모두**에 붙여넣어야 잡이 안 겹친다.

이 노트북이 `B` = 학습으로 배정되면, 아래 eval 셀들은 자동으로 비고 8)번 학습 셀만 쓰면 된다.

In [ ]:
roles = X.suggest()

# 위 출력이 "기본값과 다르다" 라고 하면, 알려주는 한 줄을 **세 노트북 모두**에 붙여넣고
# 이 셀 아래를 다시 실행할 것. (세 노드가 같은 역할표를 봐야 잡이 안 겹친다)
# X.ROLE_OVERRIDE = {'C': 'train'}


## 4) 오늘 밤 계획

예상 소요시간까지 찍는다. 12시간을 넘으면 아침에 이어서 돌면 된다.

In [ ]:
plan = X.plan('B', GPUS)


## 5) dry-run — 커맨드 확인

In [ ]:
X.run_evals(plan['eval'][:2], plan['gpus'], dry=True)


## 6) preflight (12분) — **한 번만, 셋 중 한 노드에서만**

`--policy.n_action_steps` / `--policy.temporal_ensemble_coeff` override가 `lerobot_eval`에서 실제로 먹는지 확인한다. 여기서 죽으면 override 경로가 막힌 것이고, 그러면 조합별 학습이 필요해져 **계획을 전면 수정**해야 한다. 12시간 걸기 전에 확인할 값어치가 있다.

다른 노드에서 이미 통과했으면 이 셀은 건너뛸 것.

In [ ]:
X.preflight(gpu=plan['gpus'][0], n_ep=5)


## 7) 실행 — eval

`GPUS` 수만큼 청크로 돌고 청크마다 블로킹한다. 로그는 `outputs/final/_logs/exp5__*.log`.
아침에 이 셀을 다시 실행하면 완료분은 skip되고 남은 것부터 이어서 돈다.


In [ ]:
X.run_evals(plan['eval'], plan['gpus'])


## 8) 학습 — 이 노드가 `train`으로 배정됐거나, eval 큐가 말랐을 때만

`plan['train']`은 다른 노드와 겹치지 않는 결정론적 슬라이스다 (학습 노드들이 앞에서 2잡씩 가져가고, eval 노드는 그 뒤를 집는다).

**eval이 아직 남아 있는데 여기를 실행하면 eval이 통째로 15시간 밀린다.** 3)번에서 `train`으로 배정된 게 아니면 주석 그대로 둘 것.

먼저 dry-run으로 `--use_chunk_pairs` / `--policy.sscp_enabled` 를 확인하고 실행.

In [ ]:
X.run_trains(plan['train'], plan['gpus'], dry=True)
# X.run_trains(plan['train'], plan['gpus'])


## 아침에 볼 것

`X.report()` 가 TE 표 + 레짐 맵을 전부 찍는다. 판단 기준:

| 결과 | 다음 |
|---|---|
| `act+TE@K50` **<** `bimamba+TE@K50` | 크로스오버가 K<50 → ACT K=20/15/10 학습이 급함 (노드 A 내일 큐) |
| `act+TE@K50` **>** `bimamba+TE@K50` | **크로스오버 = K 50~100 확정.** 논문을 "long-chunk regime"으로 리라이트 시작 |
| 레짐 맵에서 `bimamba_cpoff`가 긴 stride에서 `act` 상회 | `bimamba_pure`(A에서 학습중)로 확정 → seed 1,2 추가 |
| `carry`가 짧은 stride에서 `act` 상회 | 레짐 논문 확정 ("두 메커니즘은 보완재가 아니라 대체재") |
| 아무것도 ACT를 못 이김 | TE 축이 유일한 카드 → 거기로 올인 |

**주의**: seed 1개 · 500ep 기준 binomial SE ≈ ±2.2%p. **5%p 미만 차이는 주장하지 말 것.**


In [ ]:
X.report()
